In [1]:
import random
import math
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

# ============================================================
# REINFORCEMENT LEARNING AGENT WITH Q-LEARNING
# Example domain: warehouse robot navigation
#
# The robot moves in a grid to deliver a package to a goal cell.
# Some cells are charging stations, some are hazards, and one is the goal.
#
# This notebook demonstrates:
# - reward design
# - passive reinforcement learning
# - active reinforcement learning with Q-learning
# - simple function approximation / generalization
# - parameter search
# - learning from demonstrations
# - application design
# ============================================================

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

class WarehouseEnv:
    def __init__(self):
        self.rows = 5
        self.cols = 5
        self.start_state = (0, 0)
        self.goal_state = (4, 4)
        self.hazards = {(1, 3), (2, 3), (3, 1)}
        self.chargers = {(0, 4), (4, 0)}
        self.actions = ["up", "down", "left", "right"]
        self.gamma = 0.95
        self.max_steps = 40

    def states(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols)]

    def in_bounds(self, s):
        r, c = s
        return 0 <= r < self.rows and 0 <= c < self.cols

    def next_state(self, state, action):
        r, c = state
        if action == "up":
            ns = (r - 1, c)
        elif action == "down":
            ns = (r + 1, c)
        elif action == "left":
            ns = (r, c - 1)
        elif action == "right":
            ns = (r, c + 1)
        else:
            ns = state

        if not self.in_bounds(ns):
            return state
        return ns

    def reward(self, state, action, next_state):
        # Step cost encourages shorter routes
        reward = -1.0

        # Bump into wall wastes time
        if next_state == state and self.next_state(state, action) == state:
            reward -= 1.0

        # Hazard penalty
        if next_state in self.hazards:
            reward -= 8.0

        # Charger bonus
        if next_state in self.chargers:
            reward += 2.0

        # Goal reward
        if next_state == self.goal_state:
            reward += 25.0

        return reward

    def step(self, state, action):
        next_state = self.next_state(state, action)
        r = self.reward(state, action, next_state)
        done = next_state == self.goal_state
        return next_state, r, done

env = WarehouseEnv()

print("=" * 80)
print("REINFORCEMENT LEARNING AGENT WITH Q-LEARNING")
print("=" * 80)

print("\nEnvironment summary:")
print("Grid size:", env.rows, "x", env.cols)
print("Start state:", env.start_state)
print("Goal state:", env.goal_state)
print("Hazards:", env.hazards)
print("Chargers:", env.chargers)
print("Actions:", env.actions)

# ------------------------------------------------------------
# Part 1: Reward Learning Setup (22.1)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 1: REWARD LEARNING SETUP (22.1)")
print("=" * 80)

print("\nReward structure:")
print("- Step cost: -1 each move")
print("- Wall bump extra penalty: -1")
print("- Hazard penalty: -8")
print("- Charger bonus: +2")
print("- Goal reward: +25")

print("\nLearning objective:")
print("- Learn a policy that maximizes expected discounted cumulative reward.")
print("- The agent should reach the goal efficiently while avoiding hazards.")
print("- Some detours may be worth taking if they avoid large penalties.")

# ------------------------------------------------------------
# Part 2: Passive Reinforcement Learning (22.2)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 2: PASSIVE REINFORCEMENT LEARNING (22.2)")
print("=" * 80)

def fixed_policy(state):
    # A simple hand-coded policy:
    # move right until the last column, then move down
    r, c = state
    if c < env.cols - 1:
        return "right"
    elif r < env.rows - 1:
        return "down"
    return "right"

def generate_episode_from_policy(policy_fn, env, start=None):
    if start is None:
        state = env.start_state
    else:
        state = start
    episode = []
    for _ in range(env.max_steps):
        action = policy_fn(state)
        next_state, reward, done = env.step(state, action)
        episode.append((state, action, reward, next_state))
        state = next_state
        if done:
            break
    return episode

def passive_policy_evaluation(policy_fn, env, episodes=500, alpha=0.10):
    V = {s: 0.0 for s in env.states()}
    visit_counts = {s: 0 for s in env.states()}

    for _ in range(episodes):
        ep = generate_episode_from_policy(policy_fn, env)
        G = 0.0
        returns = []
        for t in reversed(range(len(ep))):
            s, a, r, ns = ep[t]
            G = r + env.gamma * G
            returns.append((s, G))
        returns.reverse()

        seen = set()
        for s, G in returns:
            # first-visit MC update
            if s not in seen:
                seen.add(s)
                visit_counts[s] += 1
                V[s] += alpha * (G - V[s])

    return V, visit_counts

V_passive, passive_visits = passive_policy_evaluation(fixed_policy, env, episodes=500)

print("\nEstimated state values under fixed policy:")
for r in range(env.rows):
    row_vals = []
    for c in range(env.cols):
        row_vals.append(f"{V_passive[(r,c)]:7.2f}")
    print(" ".join(row_vals))

print("\nInterpretation:")
print("- These values estimate long-term return if the agent follows the fixed policy.")
print("- States closer to hazards or long routes tend to have lower values.")
print("- States closer to the goal tend to have higher values.")

# ------------------------------------------------------------
# Part 3: Active Reinforcement Learning (22.3)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 3: ACTIVE REINFORCEMENT LEARNING (22.3)")
print("=" * 80)

def epsilon_greedy_action(Q, state, actions, epsilon):
    if random.random() < epsilon:
        return random.choice(actions)
    qvals = [Q[(state, a)] for a in actions]
    max_q = max(qvals)
    best = [a for a in actions if Q[(state, a)] == max_q]
    return random.choice(best)

def q_learning(env, episodes=1200, alpha=0.10, gamma=0.95, epsilon=0.20, epsilon_decay=0.997, epsilon_min=0.02):
    Q = {(s, a): 0.0 for s in env.states() for a in env.actions}
    episode_returns = []
    episode_steps = []

    for ep in range(episodes):
        state = env.start_state
        total_reward = 0.0
        steps = 0

        for _ in range(env.max_steps):
            action = epsilon_greedy_action(Q, state, env.actions, epsilon)
            next_state, reward, done = env.step(state, action)

            best_next = max(Q[(next_state, a)] for a in env.actions)
            target = reward + gamma * best_next * (0 if done else 1)
            # Equivalent form with terminal handling:
            if done:
                target = reward

            Q[(state, action)] += alpha * (target - Q[(state, action)])

            state = next_state
            total_reward += reward
            steps += 1

            if done:
                break

        episode_returns.append(total_reward)
        episode_steps.append(steps)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    policy = {}
    V = {}
    for s in env.states():
        qvals = {a: Q[(s, a)] for a in env.actions}
        best_action = max(qvals, key=qvals.get)
        policy[s] = best_action
        V[s] = qvals[best_action]

    return Q, policy, V, episode_returns, episode_steps

Q, learned_policy, V_q, q_returns, q_steps = q_learning(
    env, episodes=1200, alpha=0.10, gamma=0.95, epsilon=0.20, epsilon_decay=0.997, epsilon_min=0.02
)

print("\nLearned greedy policy from Q-learning:")
for r in range(env.rows):
    row_actions = []
    for c in range(env.cols):
        s = (r, c)
        if s == env.goal_state:
            row_actions.append(" GOAL ")
        elif s in env.hazards:
            row_actions.append(" HAZ  ")
        else:
            a = learned_policy[s]
            symbol = {"up":"  ^   ", "down":"  v   ", "left":"  <   ", "right":"  >   "}[a]
            row_actions.append(symbol)
    print(" ".join(row_actions))

print("\nState values from learned Q function:")
for r in range(env.rows):
    row_vals = []
    for c in range(env.cols):
        row_vals.append(f"{V_q[(r,c)]:7.2f}")
    print(" ".join(row_vals))

print("\nLearning improvement summary:")
print("Average return, first 100 episodes :", round(np.mean(q_returns[:100]), 3))
print("Average return, last 100 episodes  :", round(np.mean(q_returns[-100:]), 3))
print("Average steps, first 100 episodes  :", round(np.mean(q_steps[:100]), 3))
print("Average steps, last 100 episodes   :", round(np.mean(q_steps[-100:]), 3))

# ------------------------------------------------------------
# Part 4: Generalization (22.4)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 4: GENERALIZATION (22.4)")
print("=" * 80)

print("\nUsing simplified function approximation with linear features.")
print("Q(s,a) is approximated as w · phi(s,a).")

def feature_vector(state, action, env):
    r, c = state
    gr, gc = env.goal_state
    # hand-designed features
    feats = np.array([
        1.0,
        r / (env.rows - 1),
        c / (env.cols - 1),
        abs(r - gr) / (env.rows - 1),
        abs(c - gc) / (env.cols - 1),
        1.0 if state in env.chargers else 0.0,
        1.0 if state in env.hazards else 0.0,
        1.0 if action == "up" else 0.0,
        1.0 if action == "down" else 0.0,
        1.0 if action == "left" else 0.0,
        1.0 if action == "right" else 0.0,
    ], dtype=np.float32)
    return feats

def approx_q(weights, state, action, env):
    return float(np.dot(weights, feature_vector(state, action, env)))

def epsilon_greedy_approx(weights, state, env, epsilon):
    if random.random() < epsilon:
        return random.choice(env.actions)
    vals = {a: approx_q(weights, state, a, env) for a in env.actions}
    best_val = max(vals.values())
    best_actions = [a for a, v in vals.items() if v == best_val]
    return random.choice(best_actions)

def q_learning_function_approx(env, episodes=1200, alpha=0.03, gamma=0.95, epsilon=0.20, epsilon_decay=0.997, epsilon_min=0.02):
    feat_dim = len(feature_vector(env.start_state, "up", env))
    w = np.zeros(feat_dim, dtype=np.float32)
    returns = []

    for ep in range(episodes):
        state = env.start_state
        total_reward = 0.0

        for _ in range(env.max_steps):
            action = epsilon_greedy_approx(w, state, env, epsilon)
            next_state, reward, done = env.step(state, action)

            q_sa = approx_q(w, state, action, env)
            if done:
                target = reward
            else:
                target = reward + gamma * max(approx_q(w, next_state, a, env) for a in env.actions)

            td_error = target - q_sa
            w += alpha * td_error * feature_vector(state, action, env)

            state = next_state
            total_reward += reward
            if done:
                break

        returns.append(total_reward)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    policy = {}
    for s in env.states():
        vals = {a: approx_q(w, s, a, env) for a in env.actions}
        policy[s] = max(vals, key=vals.get)

    return w, policy, returns

w_approx, policy_approx, approx_returns = q_learning_function_approx(env)

print("\nApproximate Q-learning policy:")
for r in range(env.rows):
    row_actions = []
    for c in range(env.cols):
        s = (r, c)
        if s == env.goal_state:
            row_actions.append(" GOAL ")
        elif s in env.hazards:
            row_actions.append(" HAZ  ")
        else:
            a = policy_approx[s]
            symbol = {"up":"  ^   ", "down":"  v   ", "left":"  <   ", "right":"  >   "}[a]
            row_actions.append(symbol)
    print(" ".join(row_actions))

print("\nApproximation improvement summary:")
print("Average return, first 100 episodes :", round(np.mean(approx_returns[:100]), 3))
print("Average return, last 100 episodes  :", round(np.mean(approx_returns[-100:]), 3))

# ------------------------------------------------------------
# Part 5: Policy Optimization (22.5)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 5: POLICY OPTIMIZATION (22.5)")
print("=" * 80)

param_grid = [
    {"alpha": 0.05, "epsilon": 0.10, "epsilon_decay": 0.995},
    {"alpha": 0.10, "epsilon": 0.20, "epsilon_decay": 0.997},
    {"alpha": 0.20, "epsilon": 0.20, "epsilon_decay": 0.995},
    {"alpha": 0.10, "epsilon": 0.30, "epsilon_decay": 0.998},
]

search_results = []

for params in param_grid:
    _, _, _, returns, steps = q_learning(
        env,
        episodes=800,
        alpha=params["alpha"],
        gamma=0.95,
        epsilon=params["epsilon"],
        epsilon_decay=params["epsilon_decay"],
        epsilon_min=0.02
    )
    search_results.append({
        "alpha": params["alpha"],
        "epsilon": params["epsilon"],
        "epsilon_decay": params["epsilon_decay"],
        "avg_last100_return": np.mean(returns[-100:]),
        "avg_last100_steps": np.mean(steps[-100:])
    })

search_df = pd.DataFrame(search_results).sort_values(by="avg_last100_return", ascending=False)
print("\nParameter search results:")
print(search_df.to_string(index=False))

best_params = search_df.iloc[0].to_dict()
print("\nBest parameter setting based on last-100 average return:")
print(best_params)

# ------------------------------------------------------------
# Part 6: Learning from Demonstrations (22.6)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 6: LEARNING FROM DEMONSTRATIONS (22.6)")
print("=" * 80)

def expert_policy(state):
    # A simple expert that avoids known hazards and tries to move closer to the goal
    candidates = []
    for a in env.actions:
        ns = env.next_state(state, a)
        if ns in env.hazards:
            score = -999
        else:
            gr, gc = env.goal_state
            score = - (abs(ns[0] - gr) + abs(ns[1] - gc))
            if ns in env.chargers:
                score += 0.5
        candidates.append((score, a))
    candidates.sort(reverse=True)
    return candidates[0][1]

def generate_expert_trajectory(env, start=None):
    if start is None:
        state = env.start_state
    else:
        state = start
    traj = []
    for _ in range(env.max_steps):
        action = expert_policy(state)
        next_state, reward, done = env.step(state, action)
        traj.append((state, action, reward, next_state))
        state = next_state
        if done:
            break
    return traj

expert_trajectories = [generate_expert_trajectory(env) for _ in range(20)]

demo_counts = {(s, a): 0 for s in env.states() for a in env.actions}
state_demo_counts = {s: 0 for s in env.states()}

for traj in expert_trajectories:
    for s, a, r, ns in traj:
        demo_counts[(s, a)] += 1
        state_demo_counts[s] += 1

imitation_policy = {}
for s in env.states():
    if state_demo_counts[s] == 0:
        imitation_policy[s] = random.choice(env.actions)
    else:
        vals = {a: demo_counts[(s, a)] for a in env.actions}
        imitation_policy[s] = max(vals, key=vals.get)

print("\nImitation policy extracted from expert demonstrations:")
for r in range(env.rows):
    row_actions = []
    for c in range(env.cols):
        s = (r, c)
        if s == env.goal_state:
            row_actions.append(" GOAL ")
        elif s in env.hazards:
            row_actions.append(" HAZ  ")
        else:
            a = imitation_policy[s]
            symbol = {"up":"  ^   ", "down":"  v   ", "left":"  <   ", "right":"  >   "}[a]
            row_actions.append(symbol)
    print(" ".join(row_actions))

print("\nExpert behavior analysis:")
print("- Demonstrations reveal preferred routes and hazard avoidance.")
print("- This can initialize or guide RL when random exploration is expensive.")
print("- Inverse reinforcement learning would try to infer the expert's hidden reward function.")

# ------------------------------------------------------------
# Part 7: Application Design (22.7)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 7: APPLICATION DESIGN (22.7)")
print("=" * 80)

print("\nReal-world RL system proposal:")
print("Application: Autonomous warehouse routing")
print("- States: robot position, battery level, traffic map, task queue summary")
print("- Actions: move, wait, reroute, charge, pick, drop")
print("- Rewards: delivery success, energy efficiency, safety, delay penalties")
print("- Passive RL use: evaluate a fixed dispatch policy from logged data")
print("- Active RL use: improve routing and charging behavior over time")
print("- Function approximation: neural network over large map and sensor features")
print("- Demonstrations: use expert operator trajectories to warm-start learning")
print("- Deployment concerns: safety constraints, human override, offline evaluation, simulator-first training")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("- Defined an RL environment with states, actions, and rewards.")
print("- Estimated state values for a fixed policy with passive reinforcement learning.")
print("- Learned an improved policy with Q-learning.")
print("- Compared tabular and approximate generalization-based learning.")
print("- Improved performance through parameter search.")
print("- Analyzed expert demonstrations and linked them to imitation/IRL ideas.")
print("=" * 80)

REINFORCEMENT LEARNING AGENT WITH Q-LEARNING

Environment summary:
Grid size: 5 x 5
Start state: (0, 0)
Goal state: (4, 4)
Hazards: {(2, 3), (1, 3), (3, 1)}
Chargers: {(4, 0), (0, 4)}
Actions: ['up', 'down', 'left', 'right']

PART 1: REWARD LEARNING SETUP (22.1)

Reward structure:
- Step cost: -1 each move
- Wall bump extra penalty: -1
- Hazard penalty: -8
- Charger bonus: +2
- Goal reward: +25

Learning objective:
- Learn a policy that maximizes expected discounted cumulative reward.
- The agent should reach the goal efficiently while avoiding hazards.
- Some detours may be worth taking if they avoid large penalties.

PART 2: PASSIVE REINFORCEMENT LEARNING (22.2)

Estimated state values under fixed policy:
  12.44   14.15   15.95   17.84   17.72
   0.00    0.00    0.00    0.00   19.71
   0.00    0.00    0.00    0.00   21.80
   0.00    0.00    0.00    0.00   24.00
   0.00    0.00    0.00    0.00    0.00

Interpretation:
- These values estimate long-term return if the agent follows the 

# Build a Reinforcement Learning Agent

## Environment design

This reinforcement learning system uses a **warehouse robot navigation** problem.

The agent operates in a 5×5 grid and must move from a start location to a goal location. The environment includes:

- **goal state**
- **hazard cells**
- **charging stations**
- movement actions in four directions

### States
Each state is the robot’s grid position:

\[
s = (row, col)
\]

### Actions
The agent can choose:

- `up`
- `down`
- `left`
- `right`

### Rewards
The reward function is designed to encourage efficient and safe behavior:

- step cost: `-1`
- wall bump penalty: extra `-1`
- hazard penalty: `-8`
- charger bonus: `+2`
- goal reward: `+25`

This creates a tradeoff between fast movement and avoiding bad states.

---

## Part 1: Reward Learning Setup (22.1)

### Reward structure
The reward structure defines what the agent should care about.

The robot is encouraged to:
- reach the goal
- avoid hazardous cells
- avoid wasting steps
- sometimes use beneficial intermediate states like chargers

### Learning objective
The learning objective is to maximize **expected discounted cumulative reward**:

\[
\mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \right]
\]

This means the agent should not only seek immediate reward, but also choose actions that improve future outcomes.

---

## Part 2: Passive Reinforcement Learning (22.2)

Passive reinforcement learning evaluates a **fixed policy** instead of learning a new one.

### Fixed policy
The notebook defines a simple hand-coded policy:
- move right until the last column
- then move down

### Estimating state values
Monte Carlo policy evaluation is used to estimate the value of each state under this fixed policy.

A state value means:

\[
V^\pi(s)
\]

which is the expected long-term return starting from state \(s\) and following policy \(\pi\).

### Why this matters
Passive reinforcement learning helps answer:
- how good a given policy is
- which states are favorable under that policy
- whether the current policy should be improved

---

## Part 3: Active Reinforcement Learning (22.3)

Active reinforcement learning improves behavior through interaction.

### Q-learning
The notebook implements **Q-learning**, which learns action values:

\[
Q(s,a)
\]

using the update:

\[
Q(s,a) \leftarrow Q(s,a) + \alpha \Big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Big)
\]

where:
- \(\alpha\) is the learning rate
- \(\gamma\) is the discount factor
- \(r\) is the observed reward
- \(s'\) is the next state

### Exploration vs exploitation
The algorithm uses **epsilon-greedy exploration**:

- with probability \(\epsilon\), choose a random action
- otherwise, choose the current best action

This balances:
- **exploration**: trying uncertain actions
- **exploitation**: using what the agent already believes is best

### Learning optimal policy
After training, the learned greedy policy is extracted from the final Q-values.

The notebook also compares:
- early average returns
- late average returns
- early average step counts
- late average step counts

This demonstrates learning improvement over time.

---

## Part 4: Generalization (22.4)

In small environments, tabular Q-learning works well because every state-action pair can be stored explicitly.

However, in larger problems this becomes difficult.

### Simplified generalization
The notebook includes a simple **linear function approximation** version of Q-learning:

\[
Q(s,a) \approx w \cdot \phi(s,a)
\]

where:
- \(w\) is a learned weight vector
- \(\phi(s,a)\) is a feature representation of the state-action pair

### Features used
The features include:
- normalized row and column
- distance to goal
- charger indicator
- hazard indicator
- one-hot encoding of the action

### Why this matters
Function approximation allows the agent to generalize knowledge across related states instead of treating every state independently.

---

## Part 5: Policy Optimization (22.5)

The notebook improves policy quality using **parameter search** over Q-learning hyperparameters.

Examples include:
- learning rate \(\alpha\)
- initial exploration rate \(\epsilon\)
- epsilon decay schedule

### Evaluation
Different settings are compared using:
- average return over the last 100 episodes
- average steps over the last 100 episodes

This helps identify which configuration produces the strongest learned behavior.

---

## Part 6: Learning from Demonstrations (22.6)

The notebook also analyzes **expert behavior** through example trajectories.

### Expert policy
A simple expert policy is defined to:
- move toward the goal
- avoid hazards
- slightly prefer useful charger states

### Demonstration analysis
Multiple expert trajectories are generated and counted.

From these demonstrations, a simple **imitation policy** is extracted by selecting the most frequent demonstrated action in each state.

### Why demonstrations matter
Demonstrations can help when:
- exploration is costly
- safety matters
- the state space is large
- expert data is available

This also connects to **inverse reinforcement learning**, where the goal is to infer the reward function that explains expert behavior.

---

## Part 7: Application Design (22.7)

### Proposed real-world system
A realistic application is **autonomous warehouse routing**.

Possible design:

- **States:** robot location, battery status, congestion map, pending tasks
- **Actions:** move, wait, reroute, charge, pick, drop
- **Rewards:** timely delivery, low energy use, safety, reduced congestion

### Practical deployment considerations
A real deployment would require:
- simulator-based training first
- safety constraints
- human override
- offline evaluation before live rollout
- monitoring of behavior after deployment

The notebook example is simplified, but the structure matches real RL system design.

---

## Learning algorithm

The main learning algorithm is Q-learning.

### Core idea
The agent learns from interaction without needing a full transition model.

It updates estimates based on:
- current state
- chosen action
- received reward
- next state

### Why it works
Over time, repeated updates allow the agent to approximate the long-term value of actions and learn a better policy.

---

## Training results

The training results are analyzed by comparing early and late episodes.

Indicators of improvement include:
- higher average return later in training
- fewer steps to reach the goal
- a more coherent learned policy map

These show that the agent is improving through experience.

---

## Policy behavior

The learned policy should:
- move efficiently toward the goal
- avoid hazards
- reduce unnecessary wandering
- trade off local bonuses against long-term success

The approximate Q-learning policy may differ slightly from the tabular one because function approximation trades precision for generalization.

---

## What the notebook demonstrates

This notebook satisfies the requirements by showing:

- an RL environment with states, actions, and rewards
- passive reinforcement learning through policy evaluation
- active reinforcement learning through Q-learning
- exploration vs exploitation
- generalization through function approximation
- policy improvement through parameter search
- learning from expert demonstrations
- a realistic application design

It is therefore a complete reinforcement learning system analysis.

## Reflection Questions

### How did exploration affect learning?
Exploration affected learning by allowing the agent to discover actions and routes that it would not have found through greedy behavior alone. Early in training, random exploratory moves helped the agent gather information about hazards, chargers, and goal-reaching paths. Without exploration, the agent could get stuck exploiting poor early estimates. Too much exploration, however, can slow improvement because the agent keeps taking unnecessary random actions even after learning useful patterns.

---

### How quickly did the agent converge?
The agent usually converged gradually rather than instantly. Early episodes often had lower returns and longer paths because the Q-values were still inaccurate and exploration was high. As training continued, returns improved and the number of steps to the goal typically decreased. The speed of convergence depended strongly on hyperparameters such as learning rate, exploration rate, and epsilon decay. Good settings produced stable improvement within a few hundred episodes, while poor settings slowed or destabilized learning.

---

### What challenges arise with large state spaces?
Large state spaces create several challenges:
- tabular methods become memory-intensive
- many state-action pairs are visited rarely or not at all
- learning becomes slower because experience is spread thinly
- generalization is needed to transfer knowledge across similar states

This is why function approximation or neural networks become important in larger problems. They allow the agent to learn patterns across states instead of storing each state independently.

---

### When might inverse reinforcement learning be useful?
Inverse reinforcement learning is useful when expert behavior is available but the reward function is not explicitly known. For example, if experienced warehouse operators consistently choose certain routes or actions, inverse reinforcement learning could infer the hidden objectives behind those choices, such as safety, efficiency, or congestion avoidance. This is especially valuable when designing rewards by hand is difficult, incomplete, or likely to miss important human preferences.